# Notebook 03: Feature Engineering + NLP + Enrichment Externo

## Objetivo

Transformar las columnas brutas del dataset en features listas para modelado. 
Cada nueva columna tiene un **prefijo que indica su origen**:

| Prefijo | Origen | Ejemplo |
|---------|--------|---------|
| *(ninguno)* | Dato original de Raona / LinkedIn | Industry, Number of employees |
| `ai_` | Generado por la IA de enrichment de Raona | ai_SENIORITY, ai_FIT |
| `fe_` | Feature engineering propio (este notebook) | fe_company_age, fe_log_employees |
| `ext_` | Enrichment externo (este notebook) | ext_ms_maturity_score |
| `nlp_` | NLP sobre campos de texto (este notebook) | nlp_embedding_01, nlp_topic |
| `target_` | Variable objetivo | target_replied |

## Estructura del notebook

1. **Limpieza:** eliminar columnas vacias, corregir tipos
2. **Feature Engineering (fe_):** transformaciones de columnas existentes
3. **Enrichment externo (ext_):** scoring de stack Microsoft desde Technologies used
4. **NLP (nlp_):** embeddings, topics y keywords desde campos de texto
5. **Ensamblado final:** inventario completo y guardado

## Datos de entrada
- `modeling_dataset_raw.parquet` (5,987 filas, 43 columnas) - de NB01

## Datos de salida
- `modeling_dataset_final.parquet` - dataset listo para modelado en NB04

## Imports y Configuración

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
import re
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Rutas
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
WORKING_DATA = os.path.join(PROJECT_ROOT, '..', '_working', 'data')
CACHE_DIR = os.path.join(PROJECT_ROOT, '..', '_working', 'cache')
os.makedirs(CACHE_DIR, exist_ok=True)

# Cargar dataset
df = pd.read_parquet(os.path.join(WORKING_DATA, 'modeling_dataset_raw.parquet'))
print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Positivos: {df["target_replied"].sum()} ({df["target_replied"].mean():.1%})')

Dataset cargado: 5,987 filas x 48 columnas
Positivos: 841 (14.0%)


---
## 3.1 Limpieza: eliminar columnas sin información

En NB02 identificamos columnas completamente vacias (100% nulos) o con cobertura insuficiente 
para ser útiles en el modelado. Las eliminamos para simplificar el dataset.

### Criterio de eliminación:
- **100% nulos:** Gender, Revenue, Jobs posted from LinkedIn, ai_FIT_INFRA, ai_FIT_WORKPLACE, ai_FIT_MAITE
- **< 1% datos:** ai_FIT_IA (0.5%), ai_FIT_COLABORA (0.2%) - además contienen texto largo, no puntuaciones
- **Columnas de nombre** (no son features): First name, Last name, Full name

Nota: **NO eliminamos** columnas con cobertura parcial como Technologies used (15%), 
ai_CONTACT_REPORT (43%) o Contact country (54%) - estas aun pueden aportar valor predictivo.

In [2]:
# Columnas a eliminar
cols_to_drop = [
    # FIT por producto casi vacios (<1%) y son texto, no puntuaciones
    'ai_FIT_IA', 'ai_FIT_COLABORA',
    # Nombres (no son features predictivas, se usan solo para identificacion)
    'First name', 'Last name', 'Full name',
]

# Verificar que existen antes de eliminar
existing_drops = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=existing_drops)

print(f'Columnas eliminadas: {len(existing_drops)}')
for c in existing_drops:
    print(f'  - {c}')
print(f'\nDataset tras limpieza: {df.shape[0]:,} filas x {df.shape[1]} columnas')


Columnas eliminadas: 5
  - ai_FIT_IA
  - ai_FIT_COLABORA
  - First name
  - Last name
  - Full name

Dataset tras limpieza: 5,987 filas x 43 columnas


---
## 3.2 Feature Engineering (prefijo `fe_`)

Creamos nuevas features a partir de las columnas existentes. Cada transformación 
tiene una justificación de negocio o estadística.

### Resumen de features a crear:

| Feature | Fuente | Logica | Justificación |
|---------|--------|--------|---------------|
| `fe_seniority_ord` | ai_SENIORITY | Codificación ordinal CLEVEL=4...JR=0 | Jerarquia tiene orden natural |
| `fe_type_of_contact_ord` | ai_TYPE_OF_CONTACT | KEY_DECISION_MAKER=5, BUYER_CHAMPION=4, CHAMPION/INFLUENCER=3, REFERRAL=2, NULL=0 | Rol en el proceso de compra |
| `fe_fit_approved` | ai_FIT | 1 si APROBADO, 0 si no | Simplifica la variable categórica |
| `fe_fit_data_approved` | ai_FIT_DATA | 1 si SI, 0 si no | Idem para producto DATA |
| `fe_company_age` | Year founded | 2026 - Year founded | La antiguedad es más intuitiva que el año |
| `fe_log_employees` | Number of employees | log1p(valor) | Distribución muy sesgada (3 a 199K) |
| `fe_company_size_bucket` | Number of employees | Categorias: micro/small/medium/large/enterprise | Segmenta por tamaño de empresa |
| `fe_log_connections` | Number of connections | log1p(valor) | Distribución sesgada, comprime cola |
| `fe_headcount_momentum` | Growths 6mo/1yr/2yr | Medía ponderada (0.5/0.3/0.2) | Combina 3 señales de crecimiento |
| `fe_has_email` | Professional email | 1 si existe, 0 si no | Contactos con email profesional pueden ser más accesibles |
| `fe_has_bio` | Profile bio | 1 si existe, 0 si no | Perfiles completos pueden indicar mayor actividad |
| `fe_microsoft_flag` | ai_Microsoft | 1 si usa Microsoft, 0 si no | Simplifica categórica a binaria |

### 3.2.1 Codificación ordinal de ai_SENIORITY

Los niveles de seniority tienen un **orden jerárquico natural**: un C-Level tiene más poder de 
decisión que un Junior. Convertimos esta jerarquia en un número para que el modelo pueda 
capturar esta relacion.

| Valor original | Código | Significado |
|---------------|--------|-------------|
| CLEVEL | 4 | C-Suite (CEO, CTO, CFO) |
| DIRECTOR | 3 | Director de area |
| MANAGER | 2 | Manager / responsable |
| LEAD | 1 | Team lead / senior individual |
| JR | 0 | Junior / individual contributor |

In [3]:
# fe_seniority_ord: codificacion ordinal
seniority_map = {'CLEVEL': 4, 'DIRECTOR': 3, 'MANAGER': 2, 'LEAD': 1, 'JR': 0}
df['fe_seniority_ord'] = df['ai_SENIORITY'].map(seniority_map)

print('=== fe_seniority_ord ===')
print(df['fe_seniority_ord'].value_counts().sort_index())
print(f'Nulos (SENIORITY no reconocido o ausente): {df["fe_seniority_ord"].isna().sum()}')

=== fe_seniority_ord ===
fe_seniority_ord
0.0    1131
1.0     994
2.0     969
3.0    1801
4.0     797
Name: count, dtype: int64
Nulos (SENIORITY no reconocido o ausente): 295


### 3.2.2 Codificación ordinal de ai_TYPE_OF_CONTACT

El campo TYPE OF CONTACT clasifica a cada contacto segun su rol en el proceso de compra:
Decisor (máximo poder de decisión), Champion/Buyer (promotor interno), 
Influencer, Referer (puede derivar), o NULL (sin clasificar).

Esta variable sustituye a Contact Score y Company Score (eliminados).

In [4]:
# fe_type_of_contact_ord: codificacion ordinal del tipo de contacto
# Limpiamos primero los emojis y variantes del campo
def clean_type_of_contact(val):
    if pd.isna(val):
        return 'NULL'
    val_clean = re.sub(r'[^a-zA-Z\s-]', '', str(val)).strip().upper()
    if 'DECISOR' in val_clean:
        return 'KEY_DECISION_MAKER'
    if 'CHAMPION' in val_clean and 'BUYER' in val_clean:
        return 'BUYER_CHAMPION'
    if 'CHAMPION' in val_clean:
        return 'CHAMPION'
    if 'INFLUENCER' in val_clean:
        return 'INFLUENCER'
    if 'REFERER' in val_clean or 'REFERIDOR' in val_clean:
        return 'REFERRAL'
    if 'NULL' in val_clean or val_clean == '':
        return 'NULL'
    return 'NULL'

df['ai_TYPE_OF_CONTACT_clean'] = df['ai_TYPE_OF_CONTACT'].apply(clean_type_of_contact)

type_of_contact_map = {
    'KEY_DECISION_MAKER': 5,
    'BUYER_CHAMPION': 4,
    'CHAMPION': 3,
    'INFLUENCER': 3,
    'REFERRAL': 2,
    'NULL': 0,
}
df['fe_type_of_contact_ord'] = df['ai_TYPE_OF_CONTACT_clean'].map(type_of_contact_map)

print('=== fe_type_of_contact_ord ===')
print('Distribucion de TYPE OF CONTACT limpio:')
print(df['ai_TYPE_OF_CONTACT_clean'].value_counts())
print(f'\nValores ordinales:')
print(df['fe_type_of_contact_ord'].value_counts().sort_index())

=== fe_type_of_contact_ord ===
Distribucion de TYPE OF CONTACT limpio:
ai_TYPE_OF_CONTACT_clean
REFERRAL              2889
INFLUENCER            1000
NULL                   802
KEY_DECISION_MAKER     605
BUYER_CHAMPION         379
CHAMPION               312
Name: count, dtype: int64

Valores ordinales:
fe_type_of_contact_ord
0     802
2    2889
3    1312
4     379
5     605
Name: count, dtype: int64


### 3.2.3 Simplificacion de ai_FIT y ai_FIT_DATA

El campo `ai_FIT` contiene valores como "APROBADO", "DESAPROBADO", "DUDA" (a veces con emojis). 
Lo simplificamos a una variable binaria: **aprobado o no**.

El campo `ai_FIT_DATA` indica si la empresa es un fit para la línea de producto DATA 
(Power BI, Fabric). Valores: SI, NO, COMPETITOR, DUDA.

In [5]:
# fe_fit_approved: 1 si APROBADO, 0 si no
def classify_fit(val):
    if pd.isna(val):
        return np.nan
    val_upper = str(val).upper()
    if 'APROBADO' in val_upper and 'DESAPROBADO' not in val_upper:
        return 1
    if val_upper.strip() == 'SI':
        return 1
    return 0

df['fe_fit_approved'] = df['ai_FIT'].apply(classify_fit)

print('=== fe_fit_approved ===')
print(df['fe_fit_approved'].value_counts())
print(f'Nulos: {df["fe_fit_approved"].isna().sum()}')

=== fe_fit_approved ===
fe_fit_approved
1.0    5384
0.0     595
Name: count, dtype: int64
Nulos: 8


In [6]:
# fe_fit_data_approved: 1 si la empresa es fit para DATA, 0 si no
fit_data_map = {'SI': 1, 'NO': 0, 'COMPETITOR': 0, 'DUDA': 0}
df['fe_fit_data_approved'] = df['ai_FIT_DATA'].map(fit_data_map)

print('=== fe_fit_data_approved ===')
print(df['fe_fit_data_approved'].value_counts())
print(f'Nulos: {df["fe_fit_data_approved"].isna().sum()}')

=== fe_fit_data_approved ===
fe_fit_data_approved
1.0    5704
0.0     275
Name: count, dtype: int64
Nulos: 8


### 3.2.4 fe_company_age (antiguedad de la empresa)

En lugar de usar el año de fundación directamente, calculamos la **antiguedad en años**. 
Esto es más interpretable: una empresa con 100 años de historia tiene un perfil muy diferente 
a una startup de 2 años.

Nota: en NB02 verificamos que los años de fundación antiguos (ej. 1156 para Hospital Santa Creu, 
1218 para Universidad de Salamanca) son **datos reales** de instituciones historicas espanolas. 
No los corregimos.

In [7]:
df['fe_company_age'] = 2026 - df['Year founded']

print('=== fe_company_age ===')
print(df['fe_company_age'].describe().round(1))
print(f'\nEmpresas mas antiguas:')
oldest = df.nlargest(5, 'fe_company_age')[['Company name', 'Year founded', 'fe_company_age']]
print(oldest.to_string(index=False))

=== fe_company_age ===
count    4151.0
mean       65.2
std        88.1
min         2.0
25%        29.0
50%        47.0
75%        65.0
max       855.0
Name: fe_company_age, dtype: float64

Empresas mas antiguas:
                                Company name  Year founded  fe_company_age
XARXA SANTA TECLA Sanitària, Social i Docent        1171.0           855.0
                    Universidad de Salamanca        1218.0           808.0
                    Universidad de Salamanca        1218.0           808.0
                      Barcelona City Council        1249.0           777.0
                      Barcelona City Council        1249.0           777.0


### 3.2.5 fe_log_employees (transformación logarítmica)

El número de empleados tiene una **distribución muy sesgada a la derecha**: la mayoria de empresas 
tiene pocos empleados, pero algunas tienen más de 100.000. La transformación logarítmica `log1p(x)` 
comprime esta cola y hace que las diferencias relativas sean proporciónales.

Usamos `log1p` (log(1+x)) en lugar de `log` para evitar problemás con valores 0.

Ejemplo: log1p(10) = 2.4, log1p(100) = 4.6, log1p(1000) = 6.9, log1p(100000) = 11.5

In [8]:
df['fe_log_employees'] = np.log1p(df['Number of employees'])

print('=== fe_log_employees ===')
print(df['fe_log_employees'].describe().round(2))
print(f'Nulos: {df["fe_log_employees"].isna().sum()}')

=== fe_log_employees ===
count    5664.00
mean        6.58
std         1.34
min         2.08
25%         5.61
50%         6.42
75%         7.39
max        12.20
Name: fe_log_employees, dtype: float64
Nulos: 323


### 3.2.6 fe_company_size_bucket (segmento por tamaño)

Agrupamos las empresas en segmentos por número de empleados. Esto permite al modelo 
capturar **diferencias no lineales** entre tamaños (ej. el comportamiento de una micropyme 
es cualitativamente diferente al de una enterprise, no solo cuantitativamente).

| Segmento | Empleados | Descripción |
|----------|-----------|-------------|
| 0 - micro | < 10 | Micropyme |
| 1 - small | 10 - 50 | Pequena empresa |
| 2 - medium | 50 - 250 | Mediana empresa |
| 3 - large | 250 - 1000 | Gran empresa |
| 4 - enterprise | > 1000 | Corporacion |

In [9]:
bins = [0, 10, 50, 250, 1000, np.inf]
labels = [0, 1, 2, 3, 4]
df['fe_company_size_bucket'] = pd.cut(
    df['Number of employees'], bins=bins, labels=labels, right=False
).astype(float)  # float para permitir NaN

size_names = {0: 'micro (<10)', 1: 'small (10-50)', 2: 'medium (50-250)', 
              3: 'large (250-1K)', 4: 'enterprise (>1K)'}

print('=== fe_company_size_bucket ===')
for k, v in sorted(size_names.items()):
    count = (df['fe_company_size_bucket'] == k).sum()
    rate = df[df['fe_company_size_bucket'] == k]['target_replied'].mean() * 100
    print(f'  {v}: {count:,} contactos ({rate:.1f}% reply rate)')
print(f'  NaN: {df["fe_company_size_bucket"].isna().sum()}')

=== fe_company_size_bucket ===
  micro (<10): 1 contactos (0.0% reply rate)
  small (10-50): 62 contactos (12.9% reply rate)
  medium (50-250): 1,218 contactos (12.6% reply rate)
  large (250-1K): 2,307 contactos (12.4% reply rate)
  enterprise (>1K): 2,076 contactos (16.6% reply rate)
  NaN: 323


### 3.2.7 fe_log_connections

Misma logica que para empleados: el número de conexiones LinkedIn tiene una distribución sesgada 
(mediana 545, max 30.000). Aplicamos log1p para normalizar.

In [10]:
df['fe_log_connections'] = np.log1p(df['Number of connections'])

print('=== fe_log_connections ===')
print(df['fe_log_connections'].describe().round(2))

=== fe_log_connections ===
count    5962.00
mean        6.13
std         1.89
min         0.00
25%         5.39
50%         6.49
75%         7.26
max        10.31
Name: fe_log_connections, dtype: float64


### 3.2.8 fe_headcount_momentum (momentum de crecimiento)

Combinamos tres señales de crecimiento de plantilla en un único indicador, 
dando **más peso al crecimiento reciente** (6 meses) que al histórico (2 años).

Formula: `0.5 * growth_6mo + 0.3 * growth_yearly + 0.2 * growth_2yr`

Empresas en crecimiento rápido pueden tener más necesidades de herramientas Microsoft.

In [11]:
df['fe_headcount_momentum'] = (
    0.5 * df['Six months headcount growth'].fillna(0) +
    0.3 * df['Yearly headcount growth'].fillna(0) +
    0.2 * df['Two years headcount growth'].fillna(0)
)

# Marcar como NaN si las tres fuentes son nulas
all_null = (df['Six months headcount growth'].isna() & 
            df['Yearly headcount growth'].isna() & 
            df['Two years headcount growth'].isna())
df.loc[all_null, 'fe_headcount_momentum'] = np.nan

# Cap outliers al percentil 1/99
if df['fe_headcount_momentum'].notna().sum() > 100:
    p1 = df['fe_headcount_momentum'].quantile(0.01)
    p99 = df['fe_headcount_momentum'].quantile(0.99)
    before_cap = df['fe_headcount_momentum'].describe()
    df['fe_headcount_momentum'] = df['fe_headcount_momentum'].clip(lower=p1, upper=p99)
    print(f'Capping aplicado: [{p1:.1f}, {p99:.1f}]')

print('=== fe_headcount_momentum ===')
print(df['fe_headcount_momentum'].describe().round(2))

Capping aplicado: [-11.7, 38.0]
=== fe_headcount_momentum ===
count    5645.00
mean        5.43
std         6.59
min       -11.70
25%         1.60
50%         4.50
75%         8.20
max        38.04
Name: fe_headcount_momentum, dtype: float64


### 3.2.9 Features binarias

Creamos indicadores simples de presencia/ausencia de información. La existencia de ciertos datos 
puede ser una señal en si misma (ej. un contacto con email profesional puede indicar un perfil 
más completo o accesible).

In [12]:
# fe_has_email: tiene email profesional
df['fe_has_email'] = df['Professional email'].notna().astype(int)

# fe_has_bio: tiene biografia en LinkedIn
df['fe_has_bio'] = df['Profile bio'].notna().astype(int)

# fe_microsoft_flag: 1 si usa Microsoft
def parse_microsoft(val):
    if pd.isna(val):
        return np.nan
    # Intentar conversion numerica primero (columna viene como float: 1.0/0.0)
    try:
        return int(float(val))
    except (ValueError, TypeError):
        pass
    val_upper = str(val).upper()
    if val_upper in ['TRUE', 'YES', 'SI']:
        return 1
    if val_upper in ['FALSE', 'NO']:
        return 0
    if 'SI' in val_upper or 'YES' in val_upper or 'MICROSOFT' in val_upper:
        return 1
    return 0

df['fe_microsoft_flag'] = df['ai_Microsoft'].apply(parse_microsoft)

print('=== Features binarias ===')
for col in ['fe_has_email', 'fe_has_bio', 'fe_microsoft_flag']:
    ones = df[col].sum()
    total = df[col].notna().sum()
    print(f'  {col}: {ones:,} positivos de {total:,} ({ones/max(total,1)*100:.1f}%)')

=== Features binarias ===
  fe_has_email: 5,391 positivos de 5,987 (90.0%)
  fe_has_bio: 3,291 positivos de 5,987 (55.0%)
  fe_microsoft_flag: 4,269.0 positivos de 5,401 (79.0%)


### 3.2.10 Resumen de features fe_ creadas

In [13]:
fe_cols = [c for c in df.columns if c.startswith('fe_')]
print(f'Total features fe_ creadas: {len(fe_cols)}')
for col in fe_cols:
    nn = df[col].notna().sum()
    print(f'  {col}: {nn:,} no-nulos ({nn/len(df)*100:.1f}%) | tipo: {df[col].dtype}')

Total features fe_ creadas: 12
  fe_seniority_ord: 5,692 no-nulos (95.1%) | tipo: float64
  fe_type_of_contact_ord: 5,987 no-nulos (100.0%) | tipo: int64
  fe_fit_approved: 5,979 no-nulos (99.9%) | tipo: float64
  fe_fit_data_approved: 5,979 no-nulos (99.9%) | tipo: float64
  fe_company_age: 4,151 no-nulos (69.3%) | tipo: float64
  fe_log_employees: 5,664 no-nulos (94.6%) | tipo: float64
  fe_company_size_bucket: 5,664 no-nulos (94.6%) | tipo: float64
  fe_log_connections: 5,962 no-nulos (99.6%) | tipo: float64
  fe_headcount_momentum: 5,645 no-nulos (94.3%) | tipo: float64
  fe_has_email: 5,987 no-nulos (100.0%) | tipo: int64
  fe_has_bio: 5,987 no-nulos (100.0%) | tipo: int64
  fe_microsoft_flag: 5,401 no-nulos (90.2%) | tipo: float64


---
## 3.3 Enrichment externo: Microsoft Stack Scoring (prefijo `ext_`)

La columna `Technologies used` contiene una lista de tecnologias que usa cada empresa 
(solo 15% de cobertura, pero para esos contactos es muy informativa).

Raona vende soluciones sobre el ecosistema Microsoft. Si una empresa ya usa tecnologias Microsoft, 
es más probable que necesite los servicios de Raona. Si usa tecnologias competidoras (Google, AWS), 
es menos probable.

### Scoring de tecnologias:

| Tecnologia | Puntos | Razon |
|-----------|--------|-------|
| Azure | +3 | Pilar fundamental del stack Microsoft |
| Microsoft 365 / Office 365 | +2 | Suite de productividad base |
| Dynamics 365 | +3 | ERP/CRM Microsoft, alto valor |
| Power Platform / Power BI | +2 | Herramientas de datos Microsoft |
| SharePoint | +2 | Colaboracion y gestion documental |
| Teams | +1 | Comúnicacion (muy comun) |
| Active Directory / Entra ID | +1 | Identidad Microsoft |
| Exchange | +1 | Email Microsoft |
| Google Workspace | -1 | Competidor directo |
| AWS | -1 | Competidor cloud |
| Salesforce | -1 | Competidor CRM |

In [14]:
# Diccionario de scoring de tecnologias
TECH_SCORES = {
    # Microsoft ecosystem (positivo)
    'azure': 3, 'microsoft azure': 3,
    'dynamics 365': 3, 'dynamics': 3,
    'microsoft 365': 2, 'office 365': 2, 'microsoft office': 2,
    'power bi': 2, 'power platform': 2, 'power apps': 2, 'power automate': 2,
    'sharepoint': 2,
    'teams': 1, 'microsoft teams': 1,
    'active directory': 1, 'entra id': 1, 'azure ad': 1,
    'exchange': 1, 'exchange online': 1,
    'windows server': 1, 'sql server': 1,
    'intune': 2, 'endpoint manager': 2,
    'copilot': 2,
    # Competidores (negativo)
    'google workspace': -1, 'google cloud': -1, 'gcp': -1,
    'aws': -1, 'amazon web services': -1,
    'salesforce': -1, 'hubspot': -1,
    'slack': -1, 'zoom': -1,
}

def score_tech_stack(tech_string):
    """Calcula el score de madurez Microsoft a partir de la lista de tecnologias."""
    if pd.isna(tech_string):
        return np.nan, np.nan
    
    tech_lower = str(tech_string).lower()
    ms_score = 0
    has_competitor = 0
    
    for tech, score in TECH_SCORES.items():
        if tech in tech_lower:
            if score > 0:
                ms_score += score
            else:
                has_competitor = 1
    
    return ms_score, has_competitor

# Aplicar scoring
tech_results = df['Technologies used'].apply(score_tech_stack)
df['ext_ms_maturity_score'] = tech_results.apply(lambda x: x[0])
df['ext_has_competitor_tech'] = tech_results.apply(lambda x: x[1])

print('=== ext_ms_maturity_score ===')
print(f'Cobertura: {df["ext_ms_maturity_score"].notna().sum():,} ({df["ext_ms_maturity_score"].notna().mean():.1%})')
print(df['ext_ms_maturity_score'].describe().round(1))

print(f'\n=== ext_has_competitor_tech ===')
print(df['ext_has_competitor_tech'].value_counts())

=== ext_ms_maturity_score ===
Cobertura: 1,006 (16.8%)
count    1006.0
mean        5.3
std         5.6
min         0.0
25%         0.0
50%         4.0
75%         8.0
max        22.0
Name: ext_ms_maturity_score, dtype: float64

=== ext_has_competitor_tech ===
ext_has_competitor_tech
1.0    610
0.0    396
Name: count, dtype: int64


In [15]:
# fe_tech_fit por producto: detectar tecnologias relevantes para cada linea de producto
def tech_contains(tech_str, keywords):
    if pd.isna(tech_str):
        return np.nan
    tech_lower = str(tech_str).lower()
    return int(any(kw in tech_lower for kw in keywords))

# Solo crear si hay cobertura suficiente (>10%)
tech_coverage = df['Technologies used'].notna().mean()
print(f'Cobertura de Technologies used: {tech_coverage:.1%}')

if tech_coverage >= 0.10:
    # Producto -> tecnologias relevantes
    product_tech_map = {
        'comunica': ['teams', 'microsoft teams', 'skype'],
        'colabora': ['sharepoint', 'onedrive', 'power apps', 'power automate', 'power platform'],
        'infra': ['azure', 'microsoft azure', 'windows server', 'sql server'],
        'ia': ['copilot', 'azure ai', 'cognitive', 'openai'],
        'data': ['power bi', 'fabric', 'synapse', 'sql server'],
        'workplace': ['microsoft 365', 'office 365', 'intune', 'endpoint manager'],
        'maite': ['entra id', 'azure ad', 'active directory', 'purview', 'intune'],
    }
    
    for product, keywords in product_tech_map.items():
        col_name = f'fe_tech_fit_{product}'
        df[col_name] = df['Technologies used'].apply(lambda x: tech_contains(x, keywords))
        hits = df[col_name].sum()
        print(f'  {col_name}: {hits} contactos con match')
else:
    print('Cobertura insuficiente para crear features de tech-product fit')

Cobertura de Technologies used: 16.8%
  fe_tech_fit_comunica: 92.0 contactos con match
  fe_tech_fit_colabora: 202.0 contactos con match
  fe_tech_fit_infra: 440.0 contactos con match
  fe_tech_fit_ia: 6.0 contactos con match
  fe_tech_fit_data: 518.0 contactos con match
  fe_tech_fit_workplace: 157.0 contactos con match
  fe_tech_fit_maite: 167.0 contactos con match


In [16]:
# Verificar: reply rate por nivel de madurez Microsoft
ms_data = df[df['ext_ms_maturity_score'].notna()].copy()
if len(ms_data) > 50:
    ms_data['ms_bucket'] = pd.cut(ms_data['ext_ms_maturity_score'], 
                                   bins=[-1, 0, 3, 6, 100], 
                                   labels=['0 (sin MS)', '1-3 (bajo)', '4-6 (medio)', '7+ (alto)'])
    grp = ms_data.groupby('ms_bucket')['target_replied'].agg(['count', 'mean']).round(3)
    grp.columns = ['contactos', 'reply_rate']
    print('=== Reply rate por madurez Microsoft ===')
    print(grp)

=== Reply rate por madurez Microsoft ===
             contactos  reply_rate
ms_bucket                         
0 (sin MS)         315       0.143
1-3 (bajo)         186       0.220
4-6 (medio)        138       0.109
7+ (alto)          367       0.169


---
## 3.4 NLP: Procesamiento de campos de texto (prefijo `nlp_`)

Los campos de texto generados por la IA de Raona contienen información rica que podemos 
convertir en features numéricas para el modelo.

### Campos disponibles:

| Campo | Cobertura | Contenido |
|-------|-----------|----------|
| `ai_COMPANY_REPORT` | 91% | Informe sobre la empresa (sector, tamaño, oportunidades) |
| `ai_CONTACT_REPORT` | 43% | Informe sobre el perfil del contacto |
| `ai_MOMENTUM` | 16% | Señales de timing (por que contactar ahora) |

### Pipeline NLP:

1. **Sentence Embeddings:** Convertimos el texto en vectores numericos de 384 dimensiones 
   usando el modelo `paraphrase-multilingual-MiniLM-L12-v2` (soporta espanol)
2. **UMAP:** Reducimos de 384 a 3 dimensiones para uso en el modelo
3. **BERTopic:** Agrupamos los textos en topics tematicos
4. **Keywords:** Extraemos señales específicas (urgencia, decisión) por conteo de palabras clave

### Nota sobre circularidad

Estos textos fueron generados por IA, no escritos por los contactos. Usar NLP sobre texto 
generado por IA podria parecer circular. Lo justificamos porque:
- La IA de Raona sintetizo información de multiples fuentes (LinkedIn, CRM, web)
- Extraemos **patrones estructurales** (longitud, temas, urgencia), no repetimos la IA
- En NB04 compararemos el rendimiento del modelo con y sin features nlp_ para medir su contribucion incremental

### 3.4.1 Features basicas de texto (longitud, presencia)

In [17]:
# nlp_report_length: longitud del informe de empresa (proxy de visibilidad/importancia)
df['nlp_report_length'] = df['ai_COMPANY_REPORT'].str.len().fillna(0)

# nlp_contact_report_length: longitud del informe de contacto
df['nlp_contact_report_length'] = df['ai_CONTACT_REPORT'].str.len().fillna(0)

# nlp_has_momentum: tiene informacion de timing
df['nlp_has_momentum'] = df['ai_MOMENTUM'].notna().astype(int)

print('=== Features basicas de texto ===')
print(f'nlp_report_length: media={df["nlp_report_length"].mean():.0f}, '
      f'mediana={df["nlp_report_length"].median():.0f}')
print(f'nlp_contact_report_length: media={df["nlp_contact_report_length"].mean():.0f}')
print(f'nlp_has_momentum: {df["nlp_has_momentum"].sum():,} contactos con momentum')

=== Features basicas de texto ===
nlp_report_length: media=7933, mediana=8058
nlp_contact_report_length: media=2187
nlp_has_momentum: 1,334 contactos con momentum


### 3.4.2 nlp_urgency_score: señales de urgencia en MOMENTUM y COMPANY REPORT

Los campos MOMENTUM y COMPANY REPORT contienen señales de por que contactar a una empresa AHORA. 
Son esencialmente el mismo tipo de información de distintas eras de la herramienta.
Buscamos palabras clave de urgencia/oportunidad en ambos campos y tomamos el máximo.

In [18]:
# Palabras clave de urgencia/oportunidad
URGENCY_KEYWORDS = [
    'crecimiento', 'expansion', 'transformacion digital', 'transformación digital',
    'nuevo proyecto', 'inversion', 'inversión', 'presupuesto',
    'contratando', 'hiring', 'growth', 'expansion',
    'licitacion', 'licitación', 'concurso',
    'migracion', 'migración', 'modernizacion', 'modernización',
    'urgente', 'inmediato', 'prioridad',
    'nuevo cto', 'nuevo cio', 'nuevo director',
]

def count_urgency_keywords(text):
    if pd.isna(text):
        return 0
    text_lower = str(text).lower()
    return sum(1 for kw in URGENCY_KEYWORDS if kw in text_lower)

# Aplicar a ambos campos (MOMENTUM y COMPANY REPORT) y tomar el maximo
df['_urgency_momentum'] = df['ai_MOMENTUM'].apply(count_urgency_keywords)
df['_urgency_report'] = df['ai_COMPANY_REPORT'].apply(count_urgency_keywords)
df['nlp_urgency_score'] = df[['_urgency_momentum', '_urgency_report']].max(axis=1)
df = df.drop(columns=['_urgency_momentum', '_urgency_report'])

print('=== nlp_urgency_score ===')
print(df['nlp_urgency_score'].value_counts().sort_index().head(10))
print(f'\nContactos con urgency > 0: {(df["nlp_urgency_score"] > 0).sum()}')

=== nlp_urgency_score ===
nlp_urgency_score
0     217
1     387
2     975
3    1451
4    1232
5     896
6     450
7     203
8     112
9      43
Name: count, dtype: int64

Contactos con urgency > 0: 5770


### 3.4.3 Sentence Embeddings con paraphrase-multilingual-MiniLM-L12-v2

Convertimos `ai_COMPANY_REPORT` (91% cobertura) en vectores numericos de 384 dimensiones. 
Este modelo:
- Soporta **espanol** (y otros 50+ idiomas)
- Produce vectores de 384 dimensiones (más eficiente que modelos de 768d)
- Captura el **significado semántico** del texto

Procesamos en batches de 500 para gestionar la memoria.

In [19]:
from sentence_transformers import SentenceTransformer

# Cargar modelo de embeddings multilingual
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings_cache = os.path.join(CACHE_DIR, 'company_report_embeddings.npy')

if os.path.exists(embeddings_cache):
    print('Cargando embeddings desde cache...')
    embeddings_full = np.load(embeddings_cache)
    print(f'Embeddings cargados: {embeddings_full.shape}')
else:
    print('Generando embeddings (puede tardar 2-5 minutos)...')
    
    # Preparar textos: reemplazar NaN con cadena vacia
    texts = df['ai_COMPANY_REPORT'].fillna('').tolist()
    
    # Generar embeddings en batches
    BATCH_SIZE = 500
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        batch_emb = embedding_model.encode(batch, show_progress_bar=False)
        all_embeddings.append(batch_emb)
        print(f'  Batch {i//BATCH_SIZE + 1}/{(len(texts)-1)//BATCH_SIZE + 1} completado')
    
    embeddings_full = np.vstack(all_embeddings)
    
    # Guardar en cache
    np.save(embeddings_cache, embeddings_full)
    print(f'\nEmbeddings generados y guardados: {embeddings_full.shape}')


/Users/acaballito/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/acaballito/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando embeddings desde cache...
Embeddings cargados: (5987, 384)


### 3.4.4 Reducción de dimensiónalidad con UMAP

384 dimensiones son demasiadas para un modelo tabular. Usamos **UMAP** (Uniform Manifold 
Appróximation and Projection) para reducir a **3 dimensiones** que preservan las relaciones 
de vecindad del espacio original.

Por qué UMAP y no PCA:
- UMAP preserva mejor la **estructura local** (clusters de empresas similares)
- PCA asume relaciones lineales; UMAP captura relaciones no lineales
- 3 dimensiones son suficientes para capturar las variaciones principales

In [20]:
import umap
import pickle

umap_cache = os.path.join(CACHE_DIR, 'umap_3d.npy')
umap_model_cache = os.path.join(CACHE_DIR, 'umap_reducer.pkl')

if os.path.exists(umap_cache) and os.path.exists(umap_model_cache):
    print('Cargando UMAP desde cache...')
    umap_3d = np.load(umap_cache)
    with open(umap_model_cache, 'rb') as f:
        umap_reducer = pickle.load(f)
else:
    print('Reduciendo dimensiones con UMAP (384 -> 3)...')
    umap_reducer = umap.UMAP(n_components=3, random_state=42, n_neighbors=15, min_dist=0.1)
    umap_3d = umap_reducer.fit_transform(embeddings_full)
    np.save(umap_cache, umap_3d)
    with open(umap_model_cache, 'wb') as f:
        pickle.dump(umap_reducer, f)
    print(f'UMAP completado. Cache guardado.')

df['nlp_embedding_01'] = umap_3d[:, 0]
df['nlp_embedding_02'] = umap_3d[:, 1]
df['nlp_embedding_03'] = umap_3d[:, 2]

print(f'Embeddings UMAP 3D: {umap_3d.shape}')


Cargando UMAP desde cache...


Embeddings UMAP 3D: (5987, 3)


### 3.4.5 Topic Modeling con BERTopic

BERTopic agrupa automáticamente los textos en **clusters tematicos** basandose en su contenido 
semántico. Cada contacto recibe un topic ID que representa el tema dominante de su empresa.

Ejemplo de topics que podrian emerger:
- Topic 0: Manufactura industrial
- Topic 1: Sector financiero
- Topic 2: Salud y hospitales
- etc.

In [21]:
from bertopic import BERTopic

topic_cache = os.path.join(CACHE_DIR, 'topic_assignments.npy')
topic_model_cache = os.path.join(CACHE_DIR, 'topic_model')

if os.path.exists(topic_cache) and os.path.exists(topic_model_cache):
    print('Cargando topics desde cache...')
    topics = np.load(topic_cache)
    topic_model = BERTopic.load(topic_model_cache)
else:
    print('Ejecutando BERTopic (puede tardar 1-2 min)...')
    texts_for_topic = df['ai_COMPANY_REPORT'].fillna('sin informacion de empresa').tolist()
    topic_model = BERTopic(
        nr_topics='auto',
        min_topic_size=30,
        language='multilingual',
        verbose=False,
    )
    topics, probs = topic_model.fit_transform(texts_for_topic, embeddings=embeddings_full)
    np.save(topic_cache, topics)
    topic_model.save(topic_model_cache)
    print(f'BERTopic completado. {len(set(topics))} topics encontrados. Cache guardado.')

df['nlp_topic'] = topics
print(f'Topics unicos: {df["nlp_topic"].nunique()}')
print(df['nlp_topic'].value_counts().head(10))


Cargando topics desde cache...


Topics unicos: 64
nlp_topic
-1    1639
 0     466
 1     399
 2     276
 3     158
 4     134
 5     121
 6     108
 7      91
 8      85
Name: count, dtype: int64


### 3.4.6 Resumen de features NLP

In [22]:
nlp_cols = [c for c in df.columns if c.startswith('nlp_')]
print(f'Total features nlp_ creadas: {len(nlp_cols)}')
for col in nlp_cols:
    nn = df[col].notna().sum()
    print(f'  {col}: {nn:,} no-nulos | tipo: {df[col].dtype}')

Total features nlp_ creadas: 8
  nlp_report_length: 5,987 no-nulos | tipo: float64
  nlp_contact_report_length: 5,987 no-nulos | tipo: float64
  nlp_has_momentum: 5,987 no-nulos | tipo: int64
  nlp_urgency_score: 5,987 no-nulos | tipo: int64
  nlp_embedding_01: 5,987 no-nulos | tipo: float32
  nlp_embedding_02: 5,987 no-nulos | tipo: float32
  nlp_embedding_03: 5,987 no-nulos | tipo: float32
  nlp_topic: 5,987 no-nulos | tipo: int64


### 3.5.0 Corrección de ai_DEPARTMENT: detectar HR y Comúnicacion

El campo ai_DEPARTMENT a veces clasifica incorrectamente a contactos de RRHH o Comúnicacion
como IT u Operations. Usamos el Job title para detectar y corregir estos casos.

In [23]:
# Corregir ai_DEPARTMENT usando Job title
hr_keywords = ['HR', 'RRHH', 'PEOPLE', 'RECURSOS HUMANOS', 'HUMAN RESOURCES', 'TALENT']
comms_keywords = ['COMUNICACION', 'COMUNICACIÓN', 'COMMUNICATIONS', 'MARKETING', 'BRAND']

def correct_department(row):
    dept = row.get('ai_DEPARTMENT', np.nan)
    title = str(row.get('Job title', '')).upper()
    
    if any(kw in title for kw in hr_keywords):
        return 'HR'
    if any(kw in title for kw in comms_keywords):
        return 'COMMUNICATIONS'
    return dept

df['fe_department_corrected'] = df.apply(correct_department, axis=1)

# Cuantos se corrigieron?
changed = (df['fe_department_corrected'] != df['ai_DEPARTMENT']).sum()
print(f'Departamentos corregidos: {changed}')
print(f'\nDistribucion corregida:')
print(df['fe_department_corrected'].value_counts())

Departamentos corregidos: 857

Distribucion corregida:
fe_department_corrected
IT                2462
OPerations        1138
Other              565
Finance            485
Sales & MKT        480
HR                 437
COMMUNICATIONS     405
Name: count, dtype: int64


---
## 3.5 Codificación de ai_DEPARTMENT (Target Encoding)

El departamento es una variable categórica con varios niveles (IT, Operations, Other, etc.). 
Usamos **target encoding**: reemplazamos cada categoria por la tasa de respuesta medía 
de ese departamento, con **suavizado** para evitar overfitting en categorias con pocas muestras.

Formula: `encoded = (count * mean_dept + global_mean * smoothing) / (count + smoothing)`

El parámetro `smoothing` (usamos 20) controla cuanto se acerca al promedio global 
cuando hay pocas muestras.

In [24]:
# Target encoding con suavizado
SMOOTHING = 20
global_mean = df['target_replied'].mean()

dept_stats = df.groupby('fe_department_corrected')['target_replied'].agg(['mean', 'count']).reset_index()
dept_stats.columns = ['ai_DEPARTMENT', 'dept_mean', 'dept_count']

# Suavizado bayesiano
dept_stats['fe_department_encoded'] = (
    (dept_stats['dept_count'] * dept_stats['dept_mean'] + SMOOTHING * global_mean) /
    (dept_stats['dept_count'] + SMOOTHING)
)

print('=== Target encoding de ai_DEPARTMENT ===')
print(f'Promedio global: {global_mean:.3f}')
print(dept_stats.sort_values('fe_department_encoded', ascending=False).to_string(index=False))

# Aplicar al dataframe
dept_map = dept_stats.set_index('ai_DEPARTMENT')['fe_department_encoded'].to_dict()
df['fe_department_encoded'] = df['fe_department_corrected'].map(dept_map)

=== Target encoding de ai_DEPARTMENT ===
Promedio global: 0.140
 ai_DEPARTMENT  dept_mean  dept_count  fe_department_encoded
COMMUNICATIONS   0.227160         405               0.223081
   Sales & MKT   0.158333         480               0.157619
         Other   0.146903         565               0.146683
    OPerations   0.142355        1138               0.142322
            IT   0.129976        2462               0.130060
            HR   0.121281         437               0.122121
       Finance   0.107216         485               0.108534


---
## 3.6 Ensamblado del dataset final

Selecciónamos todas las columnas que pasaran al modelado, organizadas por origen.

In [25]:
# Inventario final de columnas
id_cols = ['LinkedIn profile ID', 'Company URN', 'Company name']
campaign_cols = ['Campaigns', 'Campaign engagement status']
raw_cols = [
    'Job title', 'Years in role', 'Years in company', 'Number of connections',
    'Contact country', 'Geo region',
    'Industry', 'Number of employees', 'Year founded',
    'Hiring on LinkedIn',
    'Six months headcount growth', 'Two years headcount growth', 'Yearly headcount growth',
]
ai_cols = [
    'ai_SENIORITY', 'ai_DEPARTMENT', 'ai_FIT', 'ai_FIT_DATA',
    'ai_Microsoft', 'ai_TYPE_OF_CONTACT',
]
text_cols = ['ai_CONTACT_REPORT', 'ai_COMPANY_REPORT', 'ai_MOMENTUM']
fe_cols = [c for c in df.columns if c.startswith('fe_')]
ext_cols = [c for c in df.columns if c.startswith('ext_')]
nlp_cols = [c for c in df.columns if c.startswith('nlp_')]
engagement_cols = ['Conversation tags']
target_cols = ['target_replied', 'target_replied_linkedin', 'target_replied_email', 'reply_message_number']

# Columnas que NO pasan al dataset final (ya fueron transformadas)
drop_transformed = ['Profile bio', 'Professional email', 'Technologies used',
                     'Last LinkedIn post date', 'ai_FIT_clean']
drop_existing = [c for c in drop_transformed if c in df.columns]
df = df.drop(columns=drop_existing)

print('=== Inventario de columnas por origen ===')
print(f'  IDs:            {len(id_cols)}')
print(f'  Campaign:       {len(campaign_cols)}')
print(f'  Raw:            {len(raw_cols)}')
print(f'  AI-enriched:    {len(ai_cols)}')
print(f'  Text (NLP src): {len(text_cols)}')
print(f'  fe_ (eng.):     {len(fe_cols)}')
print(f'  ext_ (externo): {len(ext_cols)}')
print(f'  nlp_ (NLP):     {len(nlp_cols)}')
print(f'  Engagement:     {len(engagement_cols)}')
print(f'  Targets:        {len(target_cols)}')
total = len(id_cols) + len(campaign_cols) + len(raw_cols) + len(ai_cols) + len(text_cols) + \
        len(fe_cols) + len(ext_cols) + len(nlp_cols) + len(engagement_cols) + len(target_cols)
print(f'  TOTAL:          {total}')
print(f'  Actual df cols: {len(df.columns)}')

=== Inventario de columnas por origen ===
  IDs:            3
  Campaign:       2
  Raw:            13
  AI-enriched:    6
  Text (NLP src): 3
  fe_ (eng.):     21
  ext_ (externo): 2
  nlp_ (NLP):     8
  Engagement:     1
  Targets:        4
  TOTAL:          63
  Actual df cols: 71


In [26]:
# Tabla resumen detallada (para la memoria)
col_inventory = []
for col in df.columns:
    if col.startswith('fe_'):
        origin = 'Feature Engineering'
    elif col.startswith('ext_'):
        origin = 'Enrichment Externo'
    elif col.startswith('nlp_'):
        origin = 'NLP'
    elif col.startswith('ai_'):
        origin = 'AI-enriched (Raona)'
    elif col.startswith('target_'):
        origin = 'Target'
    elif col in id_cols:
        origin = 'ID'
    elif col in campaign_cols:
        origin = 'Campaign'
    else:
        origin = 'Raw (LinkedIn/HeyReach)'
    
    nn = df[col].notna().sum()
    col_inventory.append({
        'Columna': col,
        'Origen': origin,
        'Tipo': str(df[col].dtype),
        '% datos': f'{nn/len(df)*100:.0f}%',
    })

inv_df = pd.DataFrame(col_inventory)
print('=== Inventario completo de columnas ===')
print(inv_df.to_string(index=False))

print(f'\nResumen por origen:')
print(inv_df['Origen'].value_counts().to_string())

=== Inventario completo de columnas ===
                     Columna                  Origen    Tipo % datos
         LinkedIn profile ID                      ID  object    100%
                 Company URN                      ID float64     96%
                Company name                      ID  object     98%
                   Campaigns                Campaign  object    100%
  Campaign engagement status                Campaign  object    100%
                   Job title Raw (LinkedIn/HeyReach)  object    100%
               Years in role Raw (LinkedIn/HeyReach) float64     92%
            Years in company Raw (LinkedIn/HeyReach) float64     89%
       Number of connections Raw (LinkedIn/HeyReach) float64    100%
             Contact country Raw (LinkedIn/HeyReach)  object     59%
                  Geo region Raw (LinkedIn/HeyReach)  object    100%
                    Industry Raw (LinkedIn/HeyReach)  object     96%
         Number of employees Raw (LinkedIn/HeyReach) float64   

---
## 3.7 Guardar dataset final

In [27]:
# Guardar
output_path = os.path.join(WORKING_DATA, 'modeling_dataset_final.parquet')
df.to_parquet(output_path, index=False)
file_size = os.path.getsize(output_path) / 1e6

print(f'Guardado: modeling_dataset_final.parquet')
print(f'  Tamano: {file_size:.1f} MB')
print(f'  Filas: {df.shape[0]:,}')
print(f'  Columnas: {df.shape[1]}')

Guardado: modeling_dataset_final.parquet
  Tamano: 25.3 MB
  Filas: 5,987
  Columnas: 71


---
### Aplicar las mismas transformaciones al pool de no contactados

El pool de contactos guardado en NB01 tiene solo columnas originales. Aplicamos las mismas
transformaciones de feature engineering para que pueda ser scoreado en NB04.

In [28]:
# Cargar pool y aplicar las mismas transformaciones
pool_path = os.path.join(WORKING_DATA, 'not_contacted_pool.parquet')
df_pool = pd.read_parquet(pool_path)
print(f'Pool cargado: {len(df_pool):,} filas x {df_pool.shape[1]} columnas')

# 1. Eliminar columnas (mismas que el dataset principal)
cols_to_drop_pool = [c for c in cols_to_drop if c in df_pool.columns]
df_pool = df_pool.drop(columns=cols_to_drop_pool)

# 2. Feature engineering (misma logica que dataset principal, preservando NaN)
df_pool['fe_seniority_ord'] = df_pool['ai_SENIORITY'].map(seniority_map)  # NaN si no mapeado

if 'ai_TYPE_OF_CONTACT' in df_pool.columns:
    df_pool['ai_TYPE_OF_CONTACT_clean'] = df_pool['ai_TYPE_OF_CONTACT'].apply(clean_type_of_contact)
    df_pool['fe_type_of_contact_ord'] = df_pool['ai_TYPE_OF_CONTACT_clean'].map(type_of_contact_map)
else:
    df_pool['fe_type_of_contact_ord'] = np.nan

df_pool['fe_microsoft_flag'] = df_pool['ai_Microsoft'].apply(parse_microsoft) if 'ai_Microsoft' in df_pool.columns else 0
df_pool['fe_fit_approved'] = df_pool['ai_FIT'].apply(classify_fit) if 'ai_FIT' in df_pool.columns else np.nan

if 'ai_FIT_DATA' in df_pool.columns:
    df_pool['fe_fit_data_approved'] = df_pool['ai_FIT_DATA'].map(fit_data_map)  # NaN si no mapeado
else:
    df_pool['fe_fit_data_approved'] = np.nan

df_pool['fe_company_age'] = 2026 - df_pool['Year founded'] if 'Year founded' in df_pool.columns else np.nan
df_pool['fe_log_employees'] = np.log1p(df_pool['Number of employees']) if 'Number of employees' in df_pool.columns else np.nan

if 'Number of employees' in df_pool.columns:
    df_pool['fe_company_size_bucket'] = pd.cut(
        df_pool['Number of employees'], bins=bins, labels=labels, right=False
    ).astype(float)  # NaN preservado
else:
    df_pool['fe_company_size_bucket'] = np.nan

df_pool['fe_log_connections'] = np.log1p(df_pool['Number of connections']) if 'Number of connections' in df_pool.columns else np.nan

# Headcount momentum con mismo capping que dataset principal
df_pool['fe_headcount_momentum'] = (
    0.5 * df_pool['Six months headcount growth'].fillna(0) +
    0.3 * df_pool['Yearly headcount growth'].fillna(0) +
    0.2 * df_pool['Two years headcount growth'].fillna(0)
)
# Aplicar mismo capping (percentiles del dataset de entrenamiento)
momentum_main = df['fe_headcount_momentum'].dropna()
p1, p99 = momentum_main.quantile(0.01), momentum_main.quantile(0.99)
df_pool['fe_headcount_momentum'] = df_pool['fe_headcount_momentum'].clip(p1, p99)

df_pool['fe_has_email'] = df_pool['Professional email'].notna().astype(int) if 'Professional email' in df_pool.columns else 0
df_pool['fe_has_bio'] = df_pool['Profile bio'].notna().astype(int) if 'Profile bio' in df_pool.columns else 0

# 3. Enrichment externo
if 'Technologies used' in df_pool.columns:
    tech_results_pool = df_pool['Technologies used'].apply(score_tech_stack)
    df_pool['ext_ms_maturity_score'] = tech_results_pool.apply(lambda x: x[0])
    df_pool['ext_has_competitor_tech'] = tech_results_pool.apply(lambda x: x[1])
else:
    df_pool['ext_ms_maturity_score'] = 0
    df_pool['ext_has_competitor_tech'] = 0

# 4. Tech fit por producto
if 'Technologies used' in df_pool.columns:
    for product, keywords in product_tech_map.items():
        col_name = f'fe_tech_fit_{product}'
        df_pool[col_name] = df_pool['Technologies used'].apply(lambda x, kw=keywords: tech_contains(x, kw))

# 5. NLP features (mismas fuentes que dataset principal)
if 'ai_COMPANY_REPORT' in df_pool.columns:
    df_pool['nlp_report_length'] = df_pool['ai_COMPANY_REPORT'].str.len().fillna(0)
else:
    df_pool['nlp_report_length'] = 0

if 'ai_CONTACT_REPORT' in df_pool.columns:
    df_pool['nlp_contact_report_length'] = df_pool['ai_CONTACT_REPORT'].str.len().fillna(0)
else:
    df_pool['nlp_contact_report_length'] = 0

# nlp_has_momentum: de ai_MOMENTUM (no de ai_COMPANY_REPORT)
df_pool['nlp_has_momentum'] = df_pool['ai_MOMENTUM'].notna().astype(int) if 'ai_MOMENTUM' in df_pool.columns else 0

# nlp_urgency_score: max entre MOMENTUM y COMPANY_REPORT
if 'ai_MOMENTUM' in df_pool.columns or 'ai_COMPANY_REPORT' in df_pool.columns:
    from functools import reduce
    urgency_cols = []
    for src_col in ['ai_MOMENTUM', 'ai_COMPANY_REPORT']:
        if src_col in df_pool.columns:
            col_name = f'_urgency_{src_col}'
            df_pool[col_name] = df_pool[src_col].fillna('').apply(
                lambda x: sum(1 for kw in URGENCY_KEYWORDS if kw.lower() in str(x).lower())
            )
            urgency_cols.append(col_name)
    df_pool['nlp_urgency_score'] = df_pool[urgency_cols].max(axis=1)
    df_pool.drop(columns=urgency_cols, inplace=True)
else:
    df_pool['nlp_urgency_score'] = 0

# 6. NLP embeddings: usar modelos ya entrenados
if 'ai_COMPANY_REPORT' in df_pool.columns:
    pool_texts = df_pool['ai_COMPANY_REPORT'].fillna('').tolist()
    pool_embeddings = embedding_model.encode(pool_texts, show_progress_bar=True, batch_size=500)
    pool_umap = umap_reducer.transform(pool_embeddings)
    df_pool['nlp_embedding_01'] = pool_umap[:, 0]
    df_pool['nlp_embedding_02'] = pool_umap[:, 1]
    df_pool['nlp_embedding_03'] = pool_umap[:, 2]
    pool_texts_topic = df_pool['ai_COMPANY_REPORT'].fillna('sin informacion de empresa').tolist()
    pool_topics, _ = topic_model.transform(pool_texts_topic, pool_embeddings)
    df_pool['nlp_topic'] = pool_topics
else:
    for col in ['nlp_embedding_01', 'nlp_embedding_02', 'nlp_embedding_03', 'nlp_topic']:
        df_pool[col] = 0

# 7. Department correction + encoding (usando mismos means del dataset principal)
if 'ai_DEPARTMENT' in df_pool.columns:
    df_pool['fe_department_corrected'] = df_pool.apply(correct_department, axis=1)
    df_pool['fe_department_encoded'] = df_pool['fe_department_corrected'].map(dept_map).fillna(global_mean)
else:
    df_pool['fe_department_encoded'] = global_mean

# Guardar pool con features
pool_out_path = os.path.join(WORKING_DATA, 'not_contacted_pool.parquet')
df_pool.to_parquet(pool_out_path, index=False)
print(f'Pool con features guardado: {len(df_pool):,} filas x {df_pool.shape[1]} columnas')
print(f'  Archivo: {pool_out_path}')


Pool cargado: 4,735 filas x 75 columnas


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:  10%|█         | 1/10 [00:01<00:13,  1.48s/it]

Batches:  20%|██        | 2/10 [00:02<00:09,  1.25s/it]

Batches:  30%|███       | 3/10 [00:03<00:08,  1.19s/it]

...

Batches:  70%|███████   | 7/10 [00:07<00:03,  1.05s/it]

Batches:  80%|████████  | 8/10 [00:08<00:01,  1.00it/s]

Batches:  90%|█████████ | 9/10 [00:08<00:00,  1.34it/s]

Batches: 100%|██████████| 10/10 [00:08<00:00,  1.12it/s]

Pool con features guardado: 4,735 filas x 75 columnas
  Archivo: /Users/acaballito/Library/CloudStorage/GoogleDrive-adriana.caballero@gmail.com/.shortcut-targets-by-id/1LzrxzfxIAZukyDLfOvioF7Z2ESoOXZGz/TFM/acaballero/TFM_deliverables/../_working/data/not_contacted_pool.parquet


---
## Resumen

### Qué hemos hecho:

1. **Limpieza:** eliminamos columnas vacias o inservibles
2. **Feature Engineering (21 features fe_):** a partir de datos existentes
   - Codificación ordinal de SENIORITY y TYPE OF CONTACT
   - fe_microsoft_flag simplificado (binario)
   - fe_department_corrected: corrección de departamento usando Job title (HR, Comúnicacion)
   - Variables binarias (fit approved, has email, has bio)
   - Transformaciones numéricas (log employees, log connections, company age)
   - Combinaciones (headcount momentum, company size bucket)
   - Target encoding de departamento corregido
   - 7 features de tech-product fit por línea de producto
3. **Enrichment externo (2 features ext_):** scoring de stack tecnologico
   - Microsoft maturity score (basado en diccionario de tecnologias)
   - Flag de tecnologia competidora
4. **NLP (8 features nlp_):** derivadas de campos de texto
   - 3 dimensiones de embeddings (UMAP de sentence-transformers sobre ai_COMPANY_REPORT)
   - Topic cluster (BERTopic)
   - Longitudes de texto (company report, contact report) y score de urgencia
5. **Pool de no contactados:** procesado con las mismas transformaciones para scoring en NB04

### Dataset final: 5,987 filas x 71 columnas (31 features nuevas + 40 originales)

### Próximo paso: NB04 - Modelado (Lead Scoring, Segmentación, Recomendación)

In [29]:
# Resumen final
print('=' * 60)
print('RESUMEN NOTEBOOK 03')
print('=' * 60)
print(f'Dataset final: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'\nFeatures por origen:')
for origin in ['Raw (LinkedIn/HeyReach)', 'AI-enriched (Raona)', 
               'Feature Engineering', 'Enrichment Externo', 'NLP',
               'Target', 'ID', 'Campaign']:
    count = (inv_df['Origen'] == origin).sum()
    if count > 0:
        print(f'  {origin}: {count}')

print(f'\nArchivos generados:')
for f in sorted(os.listdir(WORKING_DATA)):
    fpath = os.path.join(WORKING_DATA, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  {f}: {size:.1f} MB')

print(f'\nArchivos en cache:')
for f in sorted(os.listdir(CACHE_DIR)):
    fpath = os.path.join(CACHE_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  {f}: {size:.1f} MB')

RESUMEN NOTEBOOK 03
Dataset final: 5,987 filas x 71 columnas

Features por origen:
  Raw (LinkedIn/HeyReach): 15
  AI-enriched (Raona): 16
  Feature Engineering: 21
  Enrichment Externo: 2
  NLP: 8
  Target: 4
  ID: 3
  Campaign: 2

Archivos generados:
  conversation_analytics_ES.parquet: 0.3 MB
  daily_analytics_ES.parquet: 0.0 MB
  metrics.json: 0.0 MB
  modeling_dataset_final.parquet: 25.3 MB
  modeling_dataset_raw.parquet: 26.9 MB
  not_contacted_pool.parquet: 11.4 MB
  not_contacted_pool_scored.parquet: 11.4 MB
  predictions.parquet: 25.4 MB
  replies_analytics_ES.parquet: 0.0 MB

Archivos en cache:
  company_report_embeddings.npy: 9.2 MB
  topic_assignments.npy: 0.0 MB
  topic_model: 39.5 MB
  umap_3d.npy: 0.1 MB
  umap_reducer.pkl: 33.6 MB
